# 03 — Corrélations avec les annotations médecin

**Objectif :** mesurer dans quelle mesure les métriques automatiques (MEDCON, COMET-QE, mBERT, Sentence-BERT) corrèlent avec le jugement humain d'un médecin sur la qualité de traduction.


## 0. Setup

In [2]:
import os

if not os.path.exists('/content/Evaluating_medical_machine_translation'):
    !git clone https://github.com/11Maroua/Evaluating_medical_machine_translation.git

os.chdir('/content/Evaluating_medical_machine_translation')
print(f'Répertoire : {os.getcwd()}')

Répertoire : /content/Evaluating_medical_machine_translation


In [3]:
from google.colab import files

os.makedirs('data', exist_ok=True)

# Uploader : dr_annotations.json + cleaned_mesh_snomed_dico.json
uploaded = files.upload()
for fname in uploaded:
    dest = f'data/{fname}'
    os.replace(fname, dest)
    print(f'  → {dest}')

Saving cleaned_mesh_snomed_dico.json to cleaned_mesh_snomed_dico.json
Saving dr_annotations.json to dr_annotations.json
  → data/cleaned_mesh_snomed_dico.json
  → data/dr_annotations.json


In [3]:
!pip install -q pyahocorasick unbabel-comet sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.0/91.0 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.4/101.4 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 75.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.7/529.7 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 1. Imports

In [4]:
import sys
import json
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from scipy.stats import pearsonr, spearmanr

os.chdir('/content/Evaluating_medical_machine_translation')
sys.path.insert(0, 'src')
from medcon import build_pairs, build_automaton, medcon_grouped

print('Imports OK')

Imports OK


## 2. Chargement des annotations médecin

In [5]:
with open('data/dr_annotations.json', encoding='utf-8') as f:
    annotations_raw = json.load(f)

medical_data = []
for doc in annotations_raw:
    if not doc['annotations']:
        continue
    ann = doc['annotations'][0]
    score, errors = None, []
    for r in ann['result']:
        if r['from_name'] == 'translation_likert':
            score = int(r['value']['choices'][0])
        elif r['from_name'] == 'document_issue_types':
            errors = r['value']['choices']
    medical_data.append({
        'source_en'      : doc['data']['question'],
        'translation_fr' : doc['data']['translation'],
        'medical_score'  : score,
        'has_term_error' : 'Terminologie incohérente' in errors,
    })

print(f'Documents annotés   : {len(medical_data)}')
print(f'Score médecin moyen : {np.mean([d["medical_score"] for d in medical_data if d["medical_score"]]):.2f}/5')
print(f'Avec erreur termo   : {sum(d["has_term_error"] for d in medical_data)} docs')

Documents annotés   : 25
Score médecin moyen : 4.32/5
Avec erreur termo   : 21 docs


In [12]:
from collections import Counter

# Distribution des scores
scores = [d['medical_score'] for d in medical_data if d['medical_score']]
score_counts = Counter(scores)

print('=== DISTRIBUTION DES SCORES MÉDECIN ===\n')
for s in sorted(score_counts):
    bar = '█' * score_counts[s]
    print(f'  {s}/5 : {bar} ({score_counts[s]} docs)')
print(f'\n  Moyenne : {np.mean(scores):.2f}  |  Min : {min(scores)}  |  Max : {max(scores)}')

# Distribution des types d'erreurs
print('\n=== TYPES D\'ERREURS ===\n')
all_errors = []
for doc in annotations_raw:
    if not doc['annotations']:
        continue
    for r in doc['annotations'][0]['result']:
        if r['from_name'] == 'document_issue_types':
            all_errors.extend(r['value']['choices'])

error_counts = Counter(all_errors)
for error, count in error_counts.most_common():
    print(f'  {error:<35} : {count} docs ({100*count/len(medical_data):.0f}%)')

print(f'\n  Docs sans erreur : {len(medical_data) - len([d for d in medical_data if any([d["has_term_error"]])])}')

=== DISTRIBUTION DES SCORES MÉDECIN ===

  4/5 : █████████████████ (17 docs)
  5/5 : ████████ (8 docs)

  Moyenne : 4.32  |  Min : 4  |  Max : 5

=== TYPES D'ERREURS ===

  Terminologie incohérente            : 21 docs (84%)
  Grammaire/Style                     : 19 docs (76%)
  Contresens                          : 1 docs (4%)

  Docs sans erreur : 4


## 3. MEDCON sur les annotations médecin

In [6]:
with open('data/cleaned_mesh_snomed_dico.json', encoding='utf-8') as f:
    raw_dict = json.load(f)

pairs        = build_pairs(raw_dict)
automaton_en = build_automaton(pairs, 'en')
automaton_fr = build_automaton(pairs, 'fr')

print('Calcul MEDCON...')
for doc in tqdm(medical_data, desc='MEDCON'):
    r = medcon_grouped(doc['source_en'], doc['translation_fr'], pairs, automaton_en, automaton_fr)
    doc['medcon_f1']        = r['f1']
    doc['medcon_precision'] = r['precision']
    doc['medcon_recall']    = r['recall']

df_medical = pd.DataFrame(medical_data)
print(f'MEDCON F1 moyen : {df_medical["medcon_f1"].mean():.3f}')

Calcul MEDCON...


MEDCON:   0%|          | 0/25 [00:00<?, ?it/s]

MEDCON F1 moyen : 0.428


## 4. COMET-QE (reference-free)

In [7]:
USE_COMET_QE = False
try:
    import torch
    from comet import download_model, load_from_checkpoint

    for model_name in ['Unbabel/wmt20-comet-qe-da', 'Unbabel/eamt22-cometinho-da']:
        try:
            print(f'Tentative : {model_name}')
            comet_qe_model = load_from_checkpoint(download_model(model_name))
            print('OK')
            break
        except:
            continue

    comet_data = [{'src': d['source_en'], 'mt': d['translation_fr']} for d in medical_data]
    comet_output = comet_qe_model.predict(
        comet_data,
        batch_size=4,
        gpus=1 if torch.cuda.is_available() else 0
    )
    for i, doc in enumerate(medical_data):
        doc['comet_qe'] = float(comet_output.scores[i])

    df_medical   = pd.DataFrame(medical_data)
    USE_COMET_QE = True
    print(f'COMET-QE moyen : {df_medical["comet_qe"].mean():.3f}')

except Exception as e:
    print(f'COMET-QE indisponible : {e}')
    for doc in medical_data:
        doc['comet_qe'] = None
    df_medical = pd.DataFrame(medical_data)

Tentative : Unbabel/wmt20-comet-qe-da


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

.gitattributes: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecat

OK


Predicting DataLoader 0: 100%|██████████| 7/7 [04:27<00:00, 38.25s/it]

COMET-QE moyen : 0.198


## 5. mBERT similarity

In [8]:
USE_MBERT = False
try:
    import torch
    from transformers import BertTokenizer, BertModel
    from sklearn.metrics.pairwise import cosine_similarity

    print('Chargement mBERT...')
    mbert_tok   = BertTokenizer.from_pretrained('bert-base-multilingual-cased')
    mbert_model = BertModel.from_pretrained('bert-base-multilingual-cased')
    mbert_model.eval()

    def get_mbert_emb(text):
        inputs = mbert_tok(text, return_tensors='pt', max_length=512, truncation=True, padding=True)
        with torch.no_grad():
            out = mbert_model(**inputs)
        return out.last_hidden_state.mean(dim=1).squeeze().numpy()

    for doc in tqdm(medical_data, desc='mBERT'):
        sim = cosine_similarity(
            [get_mbert_emb(doc['source_en'])],
            [get_mbert_emb(doc['translation_fr'])]
        )[0][0]
        doc['mbert_sim'] = float(sim)

    df_medical = pd.DataFrame(medical_data)
    USE_MBERT  = True
    print(f'mBERT similarity moyen : {df_medical["mbert_sim"].mean():.3f}')

except Exception as e:
    print(f'mBERT indisponible : {e}')
    for doc in medical_data:
        doc['mbert_sim'] = None
    df_medical = pd.DataFrame(medical_data)

Chargement mBERT...


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

mBERT:   0%|          | 0/25 [00:00<?, ?it/s]

mBERT similarity moyen : 0.851


## 6. Sentence-BERT similarity

In [9]:
USE_SBERT = False
try:
    from sentence_transformers import SentenceTransformer
    from sklearn.metrics.pairwise import cosine_similarity

    print('Chargement Sentence-BERT...')
    sbert = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')

    for doc in tqdm(medical_data, desc='Sentence-BERT'):
        sim = cosine_similarity(
            [sbert.encode(doc['source_en'])],
            [sbert.encode(doc['translation_fr'])]
        )[0][0]
        doc['sbert_sim'] = float(sim)

    df_medical = pd.DataFrame(medical_data)
    USE_SBERT  = True
    print(f'Sentence-BERT similarity moyen : {df_medical["sbert_sim"].mean():.3f}')

except Exception as e:
    print(f'Sentence-BERT indisponible : {e}')
    for doc in medical_data:
        doc['sbert_sim'] = None
    df_medical = pd.DataFrame(medical_data)

Chargement Sentence-BERT...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Sentence-BERT:   0%|          | 0/25 [00:00<?, ?it/s]

Sentence-BERT similarity moyen : 0.912


## 7. Corrélations avec le score médecin

In [11]:
print('=' * 80)
print('CORRÉLATIONS AVEC LE SCORE MÉDECIN')
print('=' * 80 + '\n')

valid_mask = df_medical['medical_score'].notna()
y = df_medical.loc[valid_mask, 'medical_score']

metrics_corr = [('MEDCON F1', 'medcon_f1')]
if USE_COMET_QE : metrics_corr.append(('COMET-QE',      'comet_qe'))
if USE_MBERT    : metrics_corr.append(('mBERT',         'mbert_sim'))
if USE_SBERT    : metrics_corr.append(('Sentence-BERT', 'sbert_sim'))

print(f"{'Métrique':<18} {'Pearson r':>10} {'p-val':>8} {'Spearman rho':>14} {'p-val':>8}")
print('-' * 64)

correlations = []
for name, col in metrics_corr:
    if col in df_medical.columns and df_medical.loc[valid_mask, col].notna().all():
        x = df_medical.loc[valid_mask, col]
        r_p, p_p = pearsonr(y, x)
        r_s, p_s = spearmanr(y, x)
        print(f'{name:<18} {r_p:>10.3f} {p_p:>8.4f} {r_s:>14.3f} {p_s:>8.4f}')
        correlations.append((name, col, r_p, p_p, r_s, p_s))


CORRÉLATIONS AVEC LE SCORE MÉDECIN

Métrique            Pearson r    p-val   Spearman rho    p-val
----------------------------------------------------------------
MEDCON F1               0.157   0.4524          0.134   0.5242
COMET-QE               -0.138   0.5113         -0.143   0.4962
mBERT                   0.255   0.2178          0.202   0.3325
Sentence-BERT          -0.087   0.6792         -0.131   0.5331
